# పాఠం 03 - ఏజెంటిక్ డిజైన్ నమూనాలు

ఈ పాఠంలో, సమర్ధవంతమైన AI ఏజెంట్లను నిర్మించడానికి మూడు ప్రాథమిక డిజైన్ నమూనాలను పరిశీలిస్తాము:

1. **స్పష్టమైన ఏజెంట్ సూచనలు** — ఏజెంట్ ప్రవర్తనను మార్గనిర్దేశం చేసే ఖచ్చితమైన, పాత్రను నిర్వచించే ప్రాంప్ట్లను రూపకల్పన చేయడం
2. **Pydantic మోడల్స్‌తో నిర్మిత అవుట్‌పుట్** — ఏజెంట్లు ఊహించదగిన, చెలామణీ చేయబడిన డేటా తిరిగి ఇస్తున్నట్లు నిర్ధారించడం
3. **ఎకైక బాధ్యత ఏజెంట్లు** — ఒక్కో పని బాగా చేసేది ఇందుకు ప్రత్యేకమైన ఏజెంట్లను డిజైన్ చేయడం

మేము ప్రతి నమూనాను **ప్రయాణ గమ్యస్థల సిఫారసు వ్యవస్థ** సందర్భంలో వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వ సాధించగల సిస్టమ్ నిర్మించడం కోసం వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా వరుసగా


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## నమూనా 1: స్పష్టమైన ఏజెంట్ సూచనలు

అత్యంత ప్రభావవంతమైన నమూనా కూడా అత్యంత సులభమైనది: మీ ఏజెంట్ కోసం స్పష్టమైన, వివరణాత్మక సూచనలను రాయడం.

మంచి సూచనలు నిర్వచిస్తాయి:
- **ఎవరు** అనే ఏజెంట్ (వ్యక్తిత్వం మరియు శైలి)
- **ఏం** చేయాలి (స్టెప్-బై-స్టెప్ బాధ్యతలు)
- **ఎలా** ప్రవర్తించాలి (పరిమితులు మరియు శైలి)

క్రింద, మేము ప్రతి స్పందనను ఆకృతీకరించే స్పష్టమైన సూచనలతో ఒక ప్రయాణ కన్సియర్జ్ ఏజెంట్‌ను సృష్టిస్తున్నాము.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## నమూనా 2: Pydantic మోడళ్ళతో నిర్మిత అవుట్పుట్  

ఉచిత-ఆకారపు టেক্স్ట్ సంభాషణకు ఉపయుక్తంగా ఉంటదు, కాని డౌన్‌స్ట్రీమ్ వ్యవస్థలు నిర్మిత డేటా అవసరం.  
**Pydantic మోడళ్ళను** ఒక **టూల్ ఫంక్షన్** తో జతచేస్తే, మనం:  

- ఏజెంట్ అవుట్పుట్ కి ఖచ్చితమైన స్కీమా నిర్వచించగలం  
- స్పందనలను ఆటోమేటిక్‌గా వాలిడేట్ చేయగలం  
- ఏజెంట్ ఫలితాలను అప్లికేషన్ లాజిక్ లో నమ్మకంగా ఇంటిగ్రేట్ చేయగలం  

అమలు యొక్క కీలకాంశం ఏజెంట్ ఆపినప్పుడు `response_format` ని అందించడం. ఇది  
మోడల్ను ఒక వాలిడేట్ అయి ఉన్న `TravelRecommendations` ఆబ్జెక్ట్ (అందుబాటులో ఉంటుంది `response.value` పై)  
ను తిరిగి ఇవ్వించమని బలవంతం చేస్తుంది, ఉచిత-ఆకారపు టেক্স్ట్ ని కాకుండా. `get_destination_details` టూల్ కూడా ఒక టైప్ చేసిన  
`DestinationRecommendation` ని తిరిగి ఇస్తుంది, కాబట్టి డేటా మొదటి నుంచే చివరి దాకా నిర్మితంగా ఉంటుంది.  


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


##  నమూనా 3: ఒకే బాధ్యత గల ఏజెంట్లు

సంక్లిష్ట పనులు ఒక్కొక్కటి ఒకే బాధ్యత ఉన్న అనేక ఫోకస్ చేయబడిన ఏజెంట్లుగా విభజింపబడినప్పుడు లాభంగా ఉంటాయి:

- ప్రదేశాలు మరియు అందుబాటుకు సంబంధించిన **డెస్టినేషన్ నిపుణుడు**
- విమానాలు, హోటళ్లు, మరియు ప్రయాణీకల్పనను నిర్వహించే **లోజిస్టిక్స్ ప్లాన్ తయారీదారు**

ఇది సాఫ్ట్వేర్ ఇంజనీరింగ్ సూత్రమైన *ప్రముఖ విషయాల వేరు పడి నిర్వహణ* — ప్రతి ఏజెంట్ ని స్వతంత్రంగా పరీక్షించడం, నిర్వహించడం, మెరుగుపరచడం సులభం అవుతుంది.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## సంగ్రహం

ఈ పాఠంలో మేము ప్రయాణ సూచక సందర్భానికి మూడు ఏజెంటిక్ డిజైన్ నమూనాలను వర్తించారు:

| నమూనా | ముఖ్య భావన | లాభం |
|---|---|---|
| **స్పష్టమైన సూచనలు** | వ్యక్తిత్వం, బాధ్యతలు మరియు పరిమితులను ముందుగానే నిర్వచించండి | సుస్థిరమైన, బ్రాండ్-ఆధారిత ఏజెంట్ ప్రవర్తన |
| **సంరచిత అవుట్‌పుట్** | ప్రతిస్పందన ఫార్మాట్‌గా Pydantic మోడళ్లను ఉపయోగించండి | ధృవీకరించబడిన, యంత్రం-చదువబడే ఫలితాలు |
| **ఒంటరి బాధ్యత** | ప్రతి ఏజెంట్‌కు ఒక కేంద్రీకృత పని ఇవ్వండి | పరీక్షించడానికి, నిర్వహించడానికి మరియు సముచితంగా రూపొందించడానికి సులభం |

ఈ నమూనాలు సహజంగా కలవుతాయి — మీరు స్పష్టమైన సూచనలను, సంరచిత అవుట్‌పుట్‌తో పాటు ఒంటరి బాధ్యత ఏజెంట్‌లో జోడించి బలమైన, ప్రొడక్షన్-రేడీ వ్యవస్థలను నిర్మించవచ్చు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
